## Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os, sys
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.decomposition import TruncatedSVD

RAW  = "/content/drive/MyDrive/Machine_learning_project/Data/Raw"
PROC = "/content/drive/MyDrive/Machine_learning_project/Data/Processed"
SRC  = "/content/drive/MyDrive/Machine_learning_project/src"

sys.path.append(SRC)
import common

target = common.load_target()
train_ids, test_ids = common.load_split()

y = target.set_index("id")["log_price"]
y_train, y_test = y.loc[train_ids].values, y.loc[test_ids].values
print("train:", len(train_ids), "| test:", len(test_ids))

train: 4926 | test: 1232


## Load the text matrices (notebook 01's output)




In [3]:
# `text_ids` defines the canonical row order for everything below.
X_tfidf = sparse.load_npz(f"{PROC}/text_tfidf.npz")     # (6158, 5002)
X_emb   = np.load(f"{PROC}/text_emb.npy")                # (6158, 770)
all_ids = np.load(f"{PROC}/text_ids.npy")               # canonical order

# text = TF-IDF + embeddings side by side (this combo was the best text-only model)
X_text_full = sparse.hstack([X_tfidf, sparse.csr_matrix(X_emb)]).tocsr()

# row helper: listing ids -> row positions
row_of = {i: k for k, i in enumerate(all_ids)}
rows   = lambda ids: [row_of[i] for i in ids]
tr_rows, te_rows = rows(train_ids), rows(test_ids)

print("text:", X_text_full.shape)

text: (6158, 5772)


## Load + encode the TABULAR block

We use `tabular_features.csv`
Encoding follows Person A's own scheme from `model.py`:
- `t`/`f` → 1/0
- `host_response_time` → **ordered** ordinal (it genuinely is ordered)
- `-1` in rate columns means "no data" → NaN → median (fit on train)
- nominal categoricals → one-hot (fit on train)

Two deliberate departures from A's per-model pipelines:
- **one-hot instead of TargetEncoder** for `neighbourhood_cleansed` — target
  encoding uses the target, which risks leakage in a shared fusion matrix.
- `estimated_revenue_l365d` is **dropped**: Inside Airbnb derives it from
  price x occupancy, so it is a **target leak** (it inflated tabular R2
  from 0.745 to 0.871).

In [4]:
# Person A shared only the pre-split files; recombine them into the full table.
tab_tr = pd.read_csv(f"{PROC}/tabular_train.csv")
tab_te = pd.read_csv(f"{PROC}/tabular_test.csv")

tab_raw = pd.concat([tab_tr, tab_te], ignore_index=True).set_index("id")
tab_raw = tab_raw.reindex(all_ids)                 # onto the canonical order

print("raw tabular:", tab_raw.shape)
print("any listings missing:", tab_raw.isna().all(axis=1).sum())

raw tabular: (6158, 50)
any listings missing: 0


In [5]:
# --- Person A's column groups (from their model.py) ---
BOOL_STR  = ["host_is_superhost", "host_has_profile_pic",
             "host_identity_verified", "instant_bookable"]
RATE_COLS = ["host_response_rate", "host_acceptance_rate"]
RESPONSE_TIME_ORDER = ["within an hour", "within a few hours", "within a day",
                       "a few days or more", "unknown"]
OHE_COLS  = ["host_verifications", "room_type", "property_type",
             "host_listing_bucket", "neighbourhood_cleansed"]
DROP_COLS = ["neighbourhood_group_cleansed",   # A drops it: redundant
             "estimated_revenue_l365d"]        # TARGET LEAK

X = tab_raw.drop(columns=[c for c in DROP_COLS if c in tab_raw.columns])

# (a) t/f -> 1/0
for c in BOOL_STR:
    if c in X.columns:
        X[c] = (X[c] == "t").astype(np.int8)

# (b) host_response_time -> ordered ordinal
if "host_response_time" in X.columns:
    oe = OrdinalEncoder(categories=[RESPONSE_TIME_ORDER],
                        handle_unknown="use_encoded_value",
                        unknown_value=len(RESPONSE_TIME_ORDER))
    X["host_response_time"] = oe.fit_transform(X[["host_response_time"]].astype(str))

# (c) -1 sentinels -> NaN -> train median
for c in RATE_COLS:
    if c in X.columns:
        X[c] = X[c].replace(-1.0, np.nan)
        X[c] = X[c].fillna(X.loc[train_ids, c].median())     # FIT: train only

# (d) one-hot the nominals — FIT ON TRAIN ONLY
ohe_present = [c for c in OHE_COLS if c in X.columns]
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
ohe.fit(X.loc[train_ids, ohe_present].astype(str))            # FIT: train only
ohe_df = pd.DataFrame(ohe.transform(X[ohe_present].astype(str)),
                      index=X.index,
                      columns=ohe.get_feature_names_out(ohe_present))
X = pd.concat([X.drop(columns=ohe_present), ohe_df], axis=1)

# (e) any leftover NaNs -> train median
if X.isna().any().any():
    X = X.fillna(X.loc[train_ids].median())

X_tab    = X.values.astype(np.float32)
tab_cols = X.columns.tolist()
print("encoded tabular:", X_tab.shape, "| NaNs:", np.isnan(X_tab).any())

encoded tabular: (6158, 152) | NaNs: False


## Load the IMAGE block

In [6]:
img_tr = pd.read_csv(f"{PROC}/train_embeddings.csv")
img_te = pd.read_csv(f"{PROC}/test_embeddings.csv")

img_all = pd.concat([img_tr, img_te]).set_index("id")
X_img_full = img_all.reindex(all_ids).values.astype(np.float32)

print("image:", X_img_full.shape, "| NaNs:", np.isnan(X_img_full).any())

image: (6158, 2049) | NaNs: False


## Compress the wide blocks

In [7]:
N_TEXT, N_IMG = 100, 50

svd_text = TruncatedSVD(n_components=N_TEXT, random_state=42).fit(X_text_full[tr_rows])
svd_img  = TruncatedSVD(n_components=N_IMG,  random_state=42).fit(X_img_full[tr_rows])

X_text = svd_text.transform(X_text_full)
X_img  = svd_img.transform(X_img_full)

print(f"text  {X_text_full.shape[1]:>5} -> {X_text.shape[1]:>3}  "
      f"(variance kept: {svd_text.explained_variance_ratio_.sum():.1%})")
print(f"image {X_img_full.shape[1]:>5} -> {X_img.shape[1]:>3}  "
      f"(variance kept: {svd_img.explained_variance_ratio_.sum():.1%})")
print(f"tabular {X_tab.shape[1]} (uncompressed)")

text   5772 -> 100  (variance kept: 78.8%)
image  2049 ->  50  (variance kept: 69.2%)
tabular 152 (uncompressed)


## Save the bundle

In [8]:
np.savez_compressed(
    f"{PROC}/feature_bundle.npz",
    X_tab=X_tab,
    X_text=X_text,
    X_img=X_img,
    all_ids=all_ids,
    train_ids=train_ids,
    test_ids=test_ids,
    y_train=y_train,
    y_test=y_test,
    tab_cols=np.array(tab_cols, dtype=object),
)
print("saved feature_bundle.npz")
print(f"  tabular {X_tab.shape} | text {X_text.shape} | image {X_img.shape}")

saved feature_bundle.npz
  tabular (6158, 152) | text (6158, 100) | image (6158, 50)
